## 1 — World and LLM Setup

In [1]:
from dataclasses import asdict
from pprint import pprint
from dotenv import load_dotenv
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body
from llmr.reasoning.llm_provider import make_llm, LLMProvider

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False
symbol_type = Body

load_dotenv('../.env')
llm = make_llm(LLMProvider.OPENAI, model='gpt-4o-mini', temperature=0.0)
print('LLM ready:', getattr(llm, 'model_name', llm))


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


LLM ready: gpt-4o-mini


In [2]:
import rclpy, threading
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()
TFPublisher(_world=world, node=_ros_node)
VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')


ROS2 publishers started


## 2 — Graph and Inference Setup

In [4]:
from krrood.entity_query_language.factories import an, entity, variable
from krrood.entity_query_language.query.match import Match
from llmr.backend import LLMBackend
from llmr.hypotheses import (
    HypothesisGraph, BuildOrchestrator,
    FlanaganGraphView, FrameNetGraphView,
    FrameClaim, RoleClaim, PlanClaim, PhaseClaim,
    Instruction, Action, ReasonerRun,
    GroundingState, ClaimStatus,
)
from llmr.hypotheses.entities.base import ClaimHypothesis, AnchorHypothesis
from llmr.hypotheses.adapters import to_dot
from llmr.reasoning.framenet_reasoner import FrameNetReasoner
from llmr.reasoning.flanagan_reasoner import FlanaganReasoner
from llmr.reasoning.wiki_exporter import WikiExporterReasoner
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction

orchestrator = BuildOrchestrator.with_default_builders()
graph = orchestrator.graph

def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=..., arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=..., vertical_alignment=..., manipulator=...,
        ),
    )

def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm, symbol_type=symbol_type, instruction=instruction,
        strict_required=True,
        reasoners=[FrameNetReasoner(llm=llm), FlanaganReasoner(llm=llm),
                   WikiExporterReasoner(wiki_dir="~/LLM_WIKIs/LLMRWiki")],
        sg_model_orchestrator=orchestrator,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend

print('Graph + inference helpers ready.')

Graph + inference helpers ready.


In [5]:
INSTRUCTION = 'pick up the milk from the table'
action, backend = run_instruction(INSTRUCTION)
print('Resolved:', type(action).__name__, '|', action.object_designator)
print(f'Graph: {len(graph)} entities')

Resolved: PickUpAction | Body(name=PrefixedName('None/milk.stl'), id=UUID('5aba3cb0-5354-4b3e-8a7c-e33eaaef2665'), index=219)
Graph: 64 entities


## 3 — Graph as EQL Domain

`HypothesisGraph` implements `__iter__` (yields all nodes). EQL's `is_iterable` path
picks this up automatically and filters candidates with `isinstance` — the same mechanism
used for plain Python lists. No more `get_instances_of_type` call at the query site.

In [ ]:
# EQL variables backed by the whole graph — type filtering is implicit
frame_v = variable(FrameClaim, domain=graph)
role_v  = variable(RoleClaim,  domain=graph)
plan_v  = variable(PlanClaim,  domain=graph)
phase_v = variable(PhaseClaim, domain=graph)

# basic sanity: same results as get_instances_of_type
assert list(an(entity(frame_v)).evaluate()) == graph.get_instances_of_type(FrameClaim)
assert list(an(entity(role_v)).evaluate())  == graph.get_instances_of_type(RoleClaim)
print('domain=graph gives same results as get_instances_of_type ✓')

In [ ]:
# Q3-a: entity roles from graph domain
entity_roles = list(an(entity(role_v).where(role_v.filler_kind == 'entity')).evaluate())
print(f'Entity-kind roles ({len(entity_roles)}):')
for r in entity_roles:
    print(f'  {r.role_name:15s}  filler={r.filler_text!r:25s}  grounding={r.meta.grounding.value}')


In [ ]:
# Q3-b: high-urgency contact phases
hot_phases = list(
    an(entity(phase_v).where(phase_v.contact == True)).evaluate()
)
print(f'High-urgency contact phases ({len(hot_phases)}):')
for p in hot_phases:
    print(f'  [{p.phase_index}] {p.phase_name:12s}')


In [ ]:
# list(
#     an(entity(phase_v).where(phase_v.contact == True, phase_v.urgency == 'high')).evaluate()
# )

## 4 — Forward Navigation from Anchor Nodes

`Instruction` and `Action` expose `.frames` / `.plans` and `.frame_claims` / `.plan_claims` properties
that walk composition links set at build time.
This makes anchor nodes the natural query entry point — all reasoner output is
reachable from the instruction or action that triggered it.

In [ ]:
# Fetch anchor nodes directly from the graph
instructions = graph.get_instances_of_type(Instruction)
actions      = graph.get_instances_of_type(Action)

print(f'Instructions in graph: {len(instructions)}')
print(f'Actions in graph     : {len(actions)}')

instr = instructions[0]
act   = actions[0]
print(f'\nInstruction: {instr.text!r}')
print(f'Action type: {act.action_type!r}')

In [ ]:
# Q4-a: forward from instruction → frames, plans
instr_frames = instr.frames   # Instruction.frames → FrameClaim list
instr_plans  = instr.plans    # Instruction.plans  → PlanClaim list

print(f'instruction.frames ({len(instr_frames)}):')
for f in instr_frames:
    print(f'  frame={f.frame!r}  lexical_unit={f.lexical_unit!r}')

print(f'\ninstruction.plans ({len(instr_plans)}):')
for p in instr_plans:
    print(f'  action_type={p.action_type!r}  phase_count={p.phase_count}')

In [ ]:
# Q4-b: forward from action → frame_claims, plan_claims
action_frames = act.frame_claims  # FrameClaim.action = this Action
action_plans  = act.plan_claims   # PlanClaim.action = this Action

print(f'action.frame_claims ({len(action_frames)}):')
for f in action_frames:
    print(f'  frame={f.frame!r}  run={f.meta.short_run_id}')

print(f'\naction.plan_claims ({len(action_plans)}):')
for p in action_plans:
    print(f'  plan action_type={p.action_type!r}  phases={p.phase_count}')

In [ ]:
# Q4-c: all phases reachable from the instruction in two hops
all_phases = [
    phase
    for plan in instr.plans
    for phase in plan.phases
]
print(f'Phases reachable from instruction ({len(all_phases)}):')
for ph in all_phases:
    print(f'  [{ph.phase_index}] {ph.phase_name:12s}  target={ph.target_object!r}  contact={ph.contact}')


## 5 — Back-Navigation from Claim Nodes

Every claim node now exposes reverse edges:

| Property | Node | Direction | Edge |
|---|---|---|---|
| `.frame` | `RoleClaimNode` | ← | `HasRoleEdge` |
| `.plan` | `PhaseClaimNode` | ← | `HasPhaseEdge` |
| `.action` | `FrameClaimNode` | → | `AboutActionEdge` |
| `.action` | `PlanClaimNode` | → | `AboutActionEdge` |

These enable multi-hop traversal starting from the leaf claim and walking toward the anchor.

In [ ]:
# Q5-a: back-navigation — role → frame
roles = graph.get_instances_of_type(RoleClaim)
theme_role = next(r for r in roles if r.role_name == 'theme')

parent_frame = theme_role.frame     # RoleClaim.frame property
print(f"role={theme_role.role_name!r}  filler={theme_role.filler_text!r}")
print(f"  → frame.frame={parent_frame.frame!r}  frame.lexical_unit={parent_frame.lexical_unit!r}")

In [ ]:
# Q5-b: back-navigation — phase → plan
phases = graph.get_instances_of_type(PhaseClaim)
grasp_phase = next(p for p in phases if p.phase_name == 'Grasp')

parent_plan = grasp_phase.plan      # PhaseClaim.plan property
print(f"phase={grasp_phase.phase_name!r}  index={grasp_phase.phase_index}")
print(f"  → plan.action_type={parent_plan.action_type!r}  plan.phase_count={parent_plan.phase_count}")

In [ ]:
# Q5-c: back-navigation — frame → action, plan → action
frames = graph.get_instances_of_type(FrameClaim)
plans  = graph.get_instances_of_type(PlanClaim)

frame_action = frames[0].action     # FrameClaim.action property
plan_action  = plans[0].action      # PlanClaim.action property

print(f"frame.action:  {frame_action.action_type!r if frame_action else None!r}")
print(f"plan.action:   {plan_action.action_type!r  if plan_action  else None!r}")
assert frame_action is plan_action, 'same Action reached from both claim types'
print('Both claim types reach the same Action ✓')

## 6 — Multi-Hop Chains

Combining graph navigation with list comprehensions to express complex queries:
start from a claim node, traverse backward to the anchor, and combine with
attributes from both ends of the chain.

In [ ]:
# Q6-a: grounded roles whose parent frame is 'Getting'
getting_grounded = [
    r for r in graph.get_instances_of_type(RoleClaim)
    if r.frame is not None
    and r.frame.frame == 'Getting'
    and r.meta.grounding == GroundingState.SYMBOL_GROUNDED
]
print(f'Grounded roles in Getting frame ({len(getting_grounded)}):')
for r in getting_grounded:
    print(f'  {r.role_name:15s}  filler={r.filler_text!r}  status={r.meta.status.value}')

In [ ]:
# Q6-b: contact phases whose plan is for PickUpAction
pickup_contact_phases = [
    ph for ph in graph.get_instances_of_type(PhaseClaim)
    if ph.contact
    and ph.plan is not None
    and ph.plan.action_type == 'PickUpAction'
]
print(f'PickUp contact phases ({len(pickup_contact_phases)}):')
for ph in pickup_contact_phases:
    print(f'  [{ph.phase_index}] {ph.phase_name:12s}  motion_type={ph.motion_type!r}  urgency={ph.urgency!r}')

In [ ]:
# Q6-c: full chain from leaf to anchor — theme filler + resolved arm
# role → frame → action → arm (action field set by slot-filling)
theme_roles = [
    r for r in graph.get_instances_of_type(RoleClaim)
    if r.role_name == 'theme' and r.frame is not None and r.frame.action is not None
]
print('Theme roles with resolved action context:')
for r in theme_roles:
    act = r.frame.action
    print(f'  theme filler={r.filler_text!r}  → action_type={act.action_type!r}')

In [ ]:
# Q6-d: cross-family — start from FrameNet role, walk to Flanagan phases
# role → frame → action → action.plan_claims → phases
def phases_for_role(role: RoleClaim):
    if role.frame is None or role.frame.action is None:
        return []
    return [
        phase
        for plan in role.frame.action.plan_claims
        for phase in plan.phases
    ]

theme_roles = [
    r for r in graph.get_instances_of_type(RoleClaim)
    if r.role_name == 'theme'
]
print('Cross-family: Flanagan phases reachable from FrameNet theme role:')
for r in theme_roles:
    related_phases = phases_for_role(r)
    print(f'  role filler={r.filler_text!r}')
    for ph in related_phases:
        print(f'    [{ph.phase_index}] {ph.phase_name}  contact={ph.contact}  urgency={ph.urgency!r}')

## 7 — Multi-Instruction Graph

When a second instruction is run on the same graph, all navigation properties
remain scoped correctly — each role, frame, phase, and plan retains its own
`_graph_ref` and can still traverse back to its own anchor.

In [ ]:
INSTRUCTION_2 = 'pick up the breakfast cereal from the counter'
action2, backend2 = run_instruction(INSTRUCTION_2)

print(f'Graph after 2 instructions: {len(graph)} entities')
print(f'FrameClaim  : {len(graph.get_instances_of_type(FrameClaim))}')
print(f'RoleClaim   : {len(graph.get_instances_of_type(RoleClaim))}')
print(f'PlanClaim   : {len(graph.get_instances_of_type(PlanClaim))}')
print(f'PhaseClaim  : {len(graph.get_instances_of_type(PhaseClaim))}')

In [ ]:
# Each role still navigates back to its own frame and action
all_roles = graph.get_instances_of_type(RoleClaim)
theme_roles = [r for r in all_roles if r.role_name == 'theme']

print(f'Theme roles across both instructions ({len(theme_roles)}):')
for r in theme_roles:
    frame = r.frame
    act   = frame.action if frame else None
    print(f'  filler={r.filler_text!r:30s}  frame={frame.frame if frame else None!r}  instruction={frame.instruction_text!r if frame else None!r}')

In [ ]:
# Q7: filter domain=graph for both instructions — theme fillers per action type
role_v2 = variable(RoleClaim, domain=graph)
all_theme_roles = list(an(entity(role_v2).where(role_v2.role_name == 'theme')).evaluate())

print(f'All theme roles from graph domain ({len(all_theme_roles)}):')
for r in all_theme_roles:
    print(f'  filler={r.filler_text!r:30s}  run={r.meta.short_run_id}')

In [ ]:
# Cross-family check: every theme role's action has at least one contact phase
results = []
for r in all_theme_roles:
    contact_phases = [
        ph for plan in r.frame.action.plan_claims
        for ph in plan.phases
        if ph.contact
    ] if r.frame and r.frame.action else []
    results.append({
        'theme_filler':    r.filler_text,
        'action_type':     r.frame.action.action_type if r.frame and r.frame.action else None,
        'contact_phases':  [ph.phase_name for ph in contact_phases],
    })

for row in results:
    print(row)

## 8 — Per-Action Scoped Queries

Scope queries to a specific action using `graph.claims_for_action(action_ref)`.
The returned list of `ClaimHypothesis` objects can be filtered with `isinstance` — no
subgraph needed since composition back-refs remain valid from any context.

In [ ]:
action1_claims = graph.claims_for_action(action)
action2_claims = graph.claims_for_action(action2)

# filter to action-specific RoleClaims
grounded_sub = [
    r for r in action1_claims
    if isinstance(r, RoleClaim) and r.meta.grounding == GroundingState.SYMBOL_GROUNDED
]

print(f'Grounded roles for action-1 ({len(grounded_sub)}):')
for r in grounded_sub:
    print(f'  {r.role_name:15s}  filler={r.filler_text!r}')

# verify back-navigation still works
for r in grounded_sub:
    assert r.frame is not None, 'frame back-nav broken'
print('Back-navigation works ✓')

In [ ]:
# Q8: both action claim sets side-by-side
for claims, label in [(action1_claims, INSTRUCTION), (action2_claims, INSTRUCTION_2)]:
    roles_in_sub  = [n for n in claims if isinstance(n, RoleClaim)]
    phases_in_sub = [n for n in claims if isinstance(n, PhaseClaim)]
    entity_roles  = [r for r in roles_in_sub if r.filler_kind == 'entity']
    contact_p     = [p for p in phases_in_sub if p.contact]
    print(f"  {label[:50]!r}")
    print(f"    roles={len(roles_in_sub)}  entity_roles={len(entity_roles)}  contact_phases={len(contact_p)}")

## 9 — Visualization

In [ ]:
try:
    from graphviz import Source
    from IPython.display import display
    display(Source(to_dot(graph), format='svg'))
except Exception as exc:
    print('Graphviz unavailable:', exc)
    print(to_dot(graph))